# Kiểm tra tài nguyên Colab

Chọn một loại runtime, kết nối rồi chạy cell bên dưới. Lưu hoặc sao chép khối JSON trước khi đổi runtime để so sánh các phiên. Bạn có thể ghi tên tùy chọn đã chọn vào `RUNTIME_LABEL` và mức CU/giờ đang hiển thị vào `CU_PER_HOUR`.

Cell đọc tên CPU, vCPU, hạn mức CPU, RAM hệ thống và GPU NVIDIA được cấp. Mỗi lần chạy là một ảnh chụp của phiên hiện tại. Cell không cần Google Drive, cài thư viện hay chạy mô hình.

In [ ]:
import csv
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import shutil
import subprocess

RUNTIME_LABEL = ""  # Ví dụ: CPU High-RAM hoặc T4
CU_PER_HOUR = None  # Điền từ mục Tài nguyên của Colab nếu muốn so chi phí

def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8").strip()
    except (OSError, UnicodeError):
        return ""

def gib(value):
    return round(value / 2**30, 2) if value is not None else None

def nvidia_gpus():
    executable = shutil.which("nvidia-smi")
    if executable is None:
        return []
    try:
        result = subprocess.run(
            [executable, "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5, check=True,
        )
    except (OSError, subprocess.SubprocessError):
        return []
    return [{"name": name.strip(), "vram_mib": int(memory) if memory.strip().isdigit() else None}
            for name, memory in csv.reader(result.stdout.splitlines())]

def runtime_inventory():
    logical = os.cpu_count()
    affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
    cpu_lines = read_text("/proc/cpuinfo").splitlines()
    model = next((line.split(":", 1)[1].strip() for line in cpu_lines if line.startswith("model name")), None)
    quota = read_text("/sys/fs/cgroup/cpu.max").split()
    cpu_limit = None
    if len(quota) == 2 and all(x.isdigit() for x in quota) and int(quota[1]) > 0:
        cpu_limit = int(quota[0]) / int(quota[1])
    meminfo = dict(line.split(":", 1) for line in read_text("/proc/meminfo").splitlines() if ":" in line)
    total = int(meminfo["MemTotal"].split()[0]) * 1024 if "MemTotal" in meminfo else None
    available = int(meminfo["MemAvailable"].split()[0]) * 1024 if "MemAvailable" in meminfo else None
    limit = read_text("/sys/fs/cgroup/memory.max")
    used = read_text("/sys/fs/cgroup/memory.current")
    limit = int(limit) if limit.isdigit() else None
    if limit is not None and used.isdigit():
        remaining = max(0, limit - int(used))
        available = min(available, remaining) if available is not None else remaining
    return {
        "runtime_label": RUNTIME_LABEL or None,
        "measured_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "cpu_model": model, "logical_vcpus": logical, "affinity_vcpus": affinity,
        "cpu_quota_vcpus": cpu_limit,
        "effective_vcpus": min((x for x in (affinity, logical, cpu_limit) if x is not None), default=None),
        "visible_ram_gib": gib(total), "ram_limit_gib": gib(limit), "available_ram_gib": gib(available),
        "nvidia_gpus": nvidia_gpus(), "compute_units_per_hour": CU_PER_HOUR,
    }

print(json.dumps(runtime_inventory(), indent=2, ensure_ascii=False))